<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **SpaceX  Falcon 9 first stage Landing Prediction**


# Lab 1: Collecting the data


Estimated time needed: **45** minutes


In this capstone, we will predict if the Falcon 9 first stage will land successfully. SpaceX advertises Falcon 9 rocket launches on its website with a cost of 62 million dollars; other providers cost upward of 165 million dollars each, much of the savings is because SpaceX can reuse the first stage. Therefore if we can determine if the first stage will land, we can determine the cost of a launch. This information can be used if an alternate company wants to bid against SpaceX for a rocket launch. In this lab, you will collect and make sure the data is in the correct format from an API. The following is an example of a successful and launch.


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/lab_v2/images/landing_1.gif)


Several examples of an unsuccessful landing are shown here:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/lab_v2/images/crash.gif)


Most unsuccessful landings are planned. Space X performs a controlled landing in the oceans. 


## Objectives


In this lab, you will make a get request to the SpaceX API. You will also do some basic data wrangling and formating. 

- Request to the SpaceX API
- Clean the requested data


----


## Import Libraries and Define Auxiliary Functions


We will import the following libraries into the lab


In [60]:
# Requests allows us to make HTTP requests which we will use to get data from an API
import requests
# Pandas is a software library written for the Python programming language for data manipulation and analysis.
import pandas as pd
# NumPy is a library for the Python programming language, adding support for large, multi-dimensional arrays and matrices, along with a large collection of high-level mathematical functions to operate on these arrays
import numpy as np
# Datetime is a library that allows us to represent dates
import datetime

# Setting this option will print all collumns of a dataframe
pd.set_option('display.max_columns', None)
# Setting this option will print all of the data in a feature
pd.set_option('display.max_colwidth', None)

Below we will define a series of helper functions that will help us use the API to extract information using identification numbers in the launch data.

From the <code>rocket</code> column we would like to learn the booster name.


In [61]:
# Takes the dataset and uses the rocket column to call the API and append the data to the list
def getBoosterVersion(data):
    for x in data['rocket']:
       if x:
        response = requests.get("https://api.spacexdata.com/v4/rockets/"+str(x)).json()
        BoosterVersion.append(response['name'])

From the <code>launchpad</code> we would like to know the name of the launch site being used, the logitude, and the latitude.


In [62]:
# Takes the dataset and uses the launchpad column to call the API and append the data to the list
def getLaunchSite(data):
    for x in data['launchpad']:
       if x:
         response = requests.get("https://api.spacexdata.com/v4/launchpads/"+str(x)).json()
         Longitude.append(response['longitude'])
         Latitude.append(response['latitude'])
         LaunchSite.append(response['name'])

From the <code>payload</code> we would like to learn the mass of the payload and the orbit that it is going to.


In [63]:
# Takes the dataset and uses the payloads column to call the API and append the data to the lists
def getPayloadData(data):
    for load in data['payloads']:
       if load:
        response = requests.get("https://api.spacexdata.com/v4/payloads/"+load).json()
        PayloadMass.append(response['mass_kg'])
        Orbit.append(response['orbit'])

From <code>cores</code> we would like to learn the outcome of the landing, the type of the landing, number of flights with that core, whether gridfins were used, wheter the core is reused, wheter legs were used, the landing pad used, the block of the core which is a number used to seperate version of cores, the number of times this specific core has been reused, and the serial of the core.


In [64]:
# Takes the dataset and uses the cores column to call the API and append the data to the lists
def getCoreData(data):
    for core in data['cores']:
            if core['core'] != None:
                response = requests.get("https://api.spacexdata.com/v4/cores/"+core['core']).json()
                Block.append(response['block'])
                ReusedCount.append(response['reuse_count'])
                Serial.append(response['serial'])
            else:
                Block.append(None)
                ReusedCount.append(None)
                Serial.append(None)
            Outcome.append(str(core['landing_success'])+' '+str(core['landing_type']))
            Flights.append(core['flight'])
            GridFins.append(core['gridfins'])
            Reused.append(core['reused'])
            Legs.append(core['legs'])
            LandingPad.append(core['landpad'])

Now let's start requesting rocket launch data from SpaceX API with the following URL:


In [65]:
spacex_url="https://api.spacexdata.com/v4/launches/past"

In [66]:
response = requests.get(spacex_url)

Check the content of the response


In [67]:
print(response.content)

b'<!DOCTYPE html>\n<!--[if lt IE 7]> <html class="no-js ie6 oldie" lang="en-US"> <![endif]-->\n<!--[if IE 7]>    <html class="no-js ie7 oldie" lang="en-US"> <![endif]-->\n<!--[if IE 8]>    <html class="no-js ie8 oldie" lang="en-US"> <![endif]-->\n<!--[if gt IE 8]><!--> <html class="no-js" lang="en-US"> <!--<![endif]-->\n<head>\n\n<title>spacexdata.com | 525: SSL handshake failed</title>\n<meta charset="UTF-8" />\n<meta http-equiv="Content-Type" content="text/html; charset=UTF-8" />\n<meta http-equiv="X-UA-Compatible" content="IE=Edge" />\n<meta name="robots" content="noindex, nofollow" />\n<meta name="viewport" content="width=device-width,initial-scale=1" />\n<link rel="stylesheet" id="cf_styles-css" href="/cdn-cgi/styles/main.css" />\n</head>\n<body>\n<div id="cf-wrapper">\n    <div id="cf-error-details" class="p-0">\n        <header class="mx-auto pt-10 lg:pt-6 lg:px-8 w-240 lg:w-full mb-8">\n            <h1 class="inline-block sm:block sm:mb-2 font-light text-60 lg:text-4xl text-bla

You should see the response contains massive information about SpaceX launches. Next, let's try to discover some more relevant information for this project.


### Task 1: Request and parse the SpaceX launch data using the GET request


To make the requested JSON results more consistent, we will use the following static response object for this project:


In [68]:
static_json_url='https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json'

We should see that the request was successfull with the 200 status response code


In [69]:
response=requests.get(static_json_url)

In [70]:
response.status_code

200

Now we decode the response content as a Json using <code>.json()</code> and turn it into a Pandas dataframe using <code>.json_normalize()</code>


In [71]:
# Use json_normalize meethod to convert the json result into a dataframe
# Autres URL possibles pour le fichier statique
static_urls = [
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json",
    "https://raw.githubusercontent.com/IBM/ds-code/master/API_call_spacex_api.json"
]

for url in static_urls:
    response = requests.get(url)
    if response.status_code == 200:
        print(f"✅ Données chargées depuis : {url}")
        res = response.json()
        df = pd.json_normalize(res)
        break
else:
    print("❌ Aucune source statique disponible. Utilisez la source GitHub ci-dessus.")

✅ Données chargées depuis : https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json


Using the dataframe <code>data</code> print the first 5 rows


In [72]:
# Get the head of the dataframe
df.head(5)

,static_fire_date_utc,static_fire_date_unix,tbd,net,window,rocket,success,details,crew,ships,capsules,payloads,launchpad,auto_update,failures,flight_number,name,date_utc,date_unix,date_local,date_precision,upcoming,cores,id,fairings.reused,fairings.recovery_attempt,fairings.recovered,fairings.ships,links.patch.small,links.patch.large,links.reddit.campaign,links.reddit.launch,links.reddit.media,links.reddit.recovery,links.flickr.small,links.flickr.original,links.presskit,links.webcast,links.youtube_id,links.article,links.wikipedia,fairings
0,2006-03-17T00:00:00.000Z,1.142554e+09,False,False,0.0,5e9d0d95eda69955f709d1eb,False,Engine failure at 33 seconds and loss of vehicle,[],[],[],[5eb0e4b5b6c3bb0006eeb1e1],5e9e4502f5090995de566f86,True,"[{'time': 33, 'altitude': None, 'reason': 'merlin engine failure'}]",1,FalconSat,2006-03-24T22:30:00.000Z,1143239400,2006-03-25T10:30:00+12:00,hour,False,"[{'core': '5e9e289df35918033d3b2623', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}]",5eb87cd9ffd86e000604b32a,False,False,False,[],https://images2.imgbox.com/3c/0e/T8iJcSN3_o.png,https://images2.imgbox.com/40/e3/GypSkayF_o.png,None,None,None,None,[],[],None,https://www.youtube.com/watch?v=0a_00nJ_Y88,0a_00nJ_Y88,https://www.space.com/2196-spacex-inaugural-falcon-1-rocket-lost-launch.html,https://en.wikipedia.org/wiki/DemoSat,NaN
1,None,NaN,False,False,0.0,5e9d0d95eda69955f709d1eb,False,"Successful first stage burn and transition to second stage, maximum altitude 289 km, Premature engine shutdown at T+7 min 30 s, Failed to reach orbit, Failed to recover first stage",[],[],[],[5eb0e4b6b6c3bb0006eeb1e2],5e9e4502f5090995de566f86,True,"[{'time': 301, 'altitude': 289, 'reason': 'harmonic oscillation leading to premature engine shutdown'}]",2,DemoSat,2007-03-21T01:10:00.000Z,1174439400,2007-03-21T13:10:00+12:00,hour,False,"[{'core': '5e9e289ef35918416a3b2624', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}]",5eb87cdaffd86e000604b32b,False,False,False,[],https://images2.imgbox.com/4f/e3/I0lkuJ2e_o.png,https://images2.imgbox.com/be/e7/iNqsqVYM_o.png,None,None,None,None,[],[],None,https://www.youtube.com/watch?v=Lk4zQ2wP-Nc,Lk4zQ2wP-Nc,https://www.space.com/3590-spacex-falcon-1-rocket-fails-reach-orbit.html,https://en.wikipedia.org/wiki/DemoSat,NaN
2,None,NaN,False,False,0.0,5e9d0d95eda69955f709d1eb,False,Residual stage 1 thrust led to collision between stage 1 and stage 2,[],[],[],"[5eb0e4b6b6c3bb0006eeb1e3, 5eb0e4b6b6c3bb0006eeb1e4]",5e9e4502f5090995de566f86,True,"[{'time': 140, 'altitude': 35, 'reason': 'residual stage-1 thrust led to collision between stage 1 and stage 2'}]",3,Trailblazer,2008-08-03T03:34:00.000Z,1217734440,2008-08-03T15:34:00+12:00,hour,False,"[{'core': '5e9e289ef3591814873b2625', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}]",5eb87cdbffd86e000604b32c,False,False,False,[],https://images2.imgbox.com/3d/86/cnu0pan8_o.png,https://images2.imgbox.com/4b/bd/d8UxLh4q_o.png,None,None,None,None,[],[],None,https://www.youtube.com/watch?v=v0w9p3U8860,v0w9p3U8860,http://www.spacex.com/news/2013/02/11/falcon-1-flight-3-mission-summary,https://en.wikipedia.org/wiki/Trailblazer_(satellite),NaN
3,2008-09-20T00:00:00.000Z,1.221869e+09,False,False,0.0,5e9d0d95eda69955f709d1eb,True,"Ratsat was carried to orbit on the first successful orbital launch of any privately funded and developed, liquid-propelled carrier rocket, the SpaceX Falcon 1",[],[],[],[5eb0e4b7b6c3bb0006eeb1e5],5e9e4502f5090995de566f86,True,[],4,RatSat,2008-09-28T23:15:00.000Z,1222643700,2008-09-28T11:15:00+12:00,hour,False,"[{'core': '5e9e289ef3591855dc3b2626', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_succes

You will notice that a lot of the data are IDs. For example the rocket column has no information about the rocket just an identification number.

We will now use the API again to get information about the launches using the IDs given for each launch. Specifically we will be using columns <code>rocket</code>, <code>payloads</code>, <code>launchpad</code>, and <code>cores</code>.


In [73]:
# 1. Sélectionner les colonnes souhaitées
data = df[['rocket', 'payloads', 'launchpad', 'cores', 'flight_number', 'date_utc']].copy()

# 2. Filtrer sur data directement (et non sur df)
data = data[data['cores'].map(len) == 1]
data = data[data['payloads'].map(len) == 1]

# 3. Extraire le premier élément des listes
data['cores'] = data['cores'].map(lambda x: x[0])
data['payloads'] = data['payloads'].map(lambda x: x[0])

# 4. Convertir la date
data['date'] = pd.to_datetime(data['date_utc']).dt.date

# 5. Filtrer par date
data = data[data['date'] <= datetime.date(2020, 11, 13)]

print(f"✅ Données traitées : {len(data)} lignes")
data.head()

✅ Données traitées : 94 lignes


,rocket,payloads,launchpad,cores,flight_number,date_utc,date
0,5e9d0d95eda69955f709d1eb,5eb0e4b5b6c3bb0006eeb1e1,5e9e4502f5090995de566f86,"{'core': '5e9e289df35918033d3b2623', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}",1,2006-03-24T22:30:00.000Z,2006-03-24
1,5e9d0d95eda69955f709d1eb,5eb0e4b6b6c3bb0006eeb1e2,5e9e4502f5090995de566f86,"{'core': '5e9e289ef35918416a3b2624', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}",2,2007-03-21T01:10:00.000Z,2007-03-21
3,5e9d0d95eda69955f709d1eb,5eb0e4b7b6c3bb0006eeb1e5,5e9e4502f5090995de566f86,"{'core': '5e9e289ef3591855dc3b2626', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}",4,2008-09-28T23:15:00.000Z,2008-09-28
4,5e9d0d95eda69955f709d1eb,5eb0e4b7b6c3bb0006eeb1e6,5e9e4502f5090995de566f86,"{'core': '5e9e289ef359184f103b2627', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}",5,2009-07-13T03:35:00.000Z,2009-07-13
5,5e9d0d95eda69973a809d1ec,5eb0e4b7b6c3bb0006eeb1e7,5e9e4501f509094ba4566f84,"{'core': '5e9e289ef359185f2b3b2628', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}",6,2010-06-04T18:45:00.000Z,2010-06-04


* From the <code>rocket</code> we would like to learn the booster name

* From the <code>payload</code> we would like to learn the mass of the payload and the orbit that it is going to

* From the <code>launchpad</code> we would like to know the name of the launch site being used, the longitude, and the latitude.

* **From <code>cores</code> we would like to learn the outcome of the landing, the type of the landing, number of flights with that core, whether gridfins were used, whether the core is reused, whether legs were used, the landing pad used, the block of the core which is a number used to seperate version of cores, the number of times this specific core has been reused, and the serial of the core.**

The data from these requests will be stored in lists and will be used to create a new dataframe.


In [85]:
#Global variables 

# --- Définir la longueur de référence ---
n = len(data)
print(f"📊 Nombre de lancements : {n}")

# --- Vérifier les listes ---
list_names = [
    'BoosterVersion', 'LaunchSite', 'Longitude', 'Latitude', 
    'PayloadMass', 'Orbit', 'Block', 'ReusedCount', 'Serial', 
    'Outcome', 'Flights', 'GridFins', 'Reused', 'Legs', 'LandingPad'
]

print("\n🔍 Vérification des listes :")
for name in list_names:
    current_len = len(globals()[name])
    if current_len != n:
        print(f"  ⚠️ {name}: {current_len} → étendue à {n}")
        if current_len < n:
            globals()[name] = globals()[name] + [None] * (n - current_len)
        else:
            globals()[name] = globals()[name][:n]
    else:
        print(f"  ✅ {name}: {current_len}")

# --- Créer le DataFrame ---
launch_dict = {
    'FlightNumber': data['flight_number'].values,
    'Date': data['date'].values,
    'BoosterVersion': BoosterVersion,
    'LaunchSite': LaunchSite,
    'Longitude': Longitude,
    'Latitude': Latitude,
    'PayloadMass': PayloadMass,
    'Orbit': Orbit,
    'Block': Block,
    'ReusedCount': ReusedCount,
    'Serial': Serial,
    'Outcome': Outcome,
    'Flights': Flights,
    'GridFins': GridFins,
    'Reused': Reused,
    'Legs': Legs,
    'LandingPad': LandingPad
}

spacex_df = pd.DataFrame(launch_dict)

print(f"\n✅ DataFrame : {len(spacex_df)} lignes, {len(spacex_df.columns)} colonnes")
print("\n📋 Aperçu :")
print(spacex_df.head())

📊 Nombre de lancements : 94

🔍 Vérification des listes :
  ✅ BoosterVersion: 94
  ⚠️ LaunchSite: 0 → étendue à 94
  ⚠️ Longitude: 0 → étendue à 94
  ⚠️ Latitude: 0 → étendue à 94
  ✅ PayloadMass: 94
  ✅ Orbit: 94
  ⚠️ Block: 0 → étendue à 94
  ⚠️ ReusedCount: 0 → étendue à 94
  ⚠️ Serial: 0 → étendue à 94
  ⚠️ Outcome: 0 → étendue à 94
  ⚠️ Flights: 0 → étendue à 94
  ⚠️ GridFins: 0 → étendue à 94
  ⚠️ Reused: 0 → étendue à 94
  ⚠️ Legs: 0 → étendue à 94
  ⚠️ LandingPad: 0 → étendue à 94

✅ DataFrame : 94 lignes, 17 colonnes

📋 Aperçu :
   FlightNumber        Date BoosterVersion LaunchSite Longitude Latitude  \
0             1  2006-03-24           None       None      None     None   
1             2  2007-03-21           None       None      None     None   
2             4  2008-09-28           None       None      None     None   
3             5  2009-07-13           None       None      None     None   
4             6  2010-06-04           None       None      None     None   



These functions will apply the outputs globally to the above variables. Let's take a looks at <code>BoosterVersion</code> variable. Before we apply  <code>getBoosterVersion</code> the list is empty:


In [88]:
#BoosterVersion
getBoosterVersion(data)
getLaunchSite(data)
getPayloadData(data)
getCoreData(data)

⚠️ Aucune donnée de rocket disponible. Vérifiez votre connexion.
⚠️ STATIC_DATA non disponible. Veuillez charger les données d'abord.
⚠️ Aucune donnée de payload disponible. Vérifiez votre connexion.
✅ 0 séries de core récupérées


Now, let's apply <code> getBoosterVersion</code> function method to get the booster version


In [99]:
# --- 1. Charger les données des rockets depuis GitHub ---
print("🔹 Chargement des données des rockets...")
url = "https://raw.githubusercontent.com/r-spacex/SpaceX-API/master/docs/rockets/v4/all.json"
response = requests.get(url)

ROCKET_NAMES = {}
if response.status_code == 200:
    rockets = response.json()
    ROCKET_NAMES = {r['id']: r['name'] for r in rockets}
    print(f"✅ {len(ROCKET_NAMES)} rockets chargées")
    for r_id, r_name in ROCKET_NAMES.items():
        print(f"  - {r_id}: {r_name}")
else:
    print(f"❌ Erreur HTTP {response.status_code}")

# --- 2. Remplir la colonne BoosterVersion dans spacex_df ---
# Extraire les IDs des rockets depuis data
rocket_ids = data['rocket'].values

# Créer la liste des noms de boosters
BoosterVersion = []
for r_id in rocket_ids:
    if r_id and r_id in ROCKET_NAMES:
        BoosterVersion.append(ROCKET_NAMES[r_id])
    else:
        BoosterVersion.append(None)

# Mettre à jour la colonne dans spacex_df
spacex_df['BoosterVersion'] = BoosterVersion

# --- 3. Vérifier les valeurs ---
print("\n🔍 Vérification des valeurs dans BoosterVersion :")
print(f"Valeurs uniques : {spacex_df['BoosterVersion'].unique()}")
print(f"Nombre de valeurs None : {spacex_df['BoosterVersion'].isna().sum()}")
print(f"Nombre de valeurs non-None : {spacex_df['BoosterVersion'].notna().sum()}")

# Compter les occurrences de chaque booster
print("\nDistribution des boosters :")
print(spacex_df['BoosterVersion'].value_counts())

# --- 4. Filtrer pour Falcon 9 ---
data_falcon9 = spacex_df[spacex_df['BoosterVersion'] == 'Falcon 9'].copy()

print(f"\n✅ Nombre de lancements Falcon 9 : {len(data_falcon9)}")
print(f"📊 Nombre de lancements total : {len(spacex_df)}")
print(f"📉 Lancements Falcon 1 supprimés : {len(spacex_df) - len(data_falcon9)}")

# --- 5. Réinitialiser l'index ---
data_falcon9.reset_index(drop=True, inplace=True)

# --- 6. Afficher les résultats ---
print("\n📋 Aperçu des 5 premières lignes :")
print(data_falcon9[['FlightNumber', 'Date', 'BoosterVersion', 'Orbit']].head())

print("\n📋 Aperçu des 5 dernières lignes :")
print(data_falcon9[['FlightNumber', 'Date', 'BoosterVersion', 'Orbit']].tail())

🔹 Chargement des données des rockets...
❌ Erreur HTTP 404

🔍 Vérification des valeurs dans BoosterVersion :
Valeurs uniques : [None]
Nombre de valeurs None : 94
Nombre de valeurs non-None : 0

Distribution des boosters :
Series([], Name: BoosterVersion, dtype: int64)

✅ Nombre de lancements Falcon 9 : 0
📊 Nombre de lancements total : 94
📉 Lancements Falcon 1 supprimés : 94

📋 Aperçu des 5 premières lignes :
Empty DataFrame
Columns: [FlightNumber, Date, BoosterVersion, Orbit]
Index: []

📋 Aperçu des 5 dernières lignes :
Empty DataFrame
Columns: [FlightNumber, Date, BoosterVersion, Orbit]
Index: []


the list has now been update 


In [100]:
BoosterVersion[0:5]

[None, None, None, None, None]

we can apply the rest of the  functions here:


In [101]:
# Call getLaunchSite
def getLaunchSite(data):
    """
    Récupère les informations des sites de lancement depuis STATIC_DATA
    """
    global Longitude, Latitude, LaunchSite
    Longitude, Latitude, LaunchSite = [], [], []
    
    # Vérifier que STATIC_DATA existe et contient les launchpads
    if 'STATIC_DATA' not in globals() or not STATIC_DATA.get('launchpads'):
        print("⚠️ STATIC_DATA non disponible. Veuillez charger les données d'abord.")
        Longitude = [None] * len(data)
        Latitude = [None] * len(data)
        LaunchSite = [None] * len(data)
        return
    
    launchpads = STATIC_DATA['launchpads']
    
    for x in data['launchpad']:
        if x and x in launchpads:
            pad = launchpads[x]
            Longitude.append(pad.get('longitude'))
            Latitude.append(pad.get('latitude'))
            LaunchSite.append(pad.get('name'))
        else:
            Longitude.append(None)
            Latitude.append(None)
            LaunchSite.append(None)
    
    print(f"✅ {len([v for v in LaunchSite if v])} sites de lancement récupérés")

In [102]:
# Call getPayloadData
# --- 1. Charger les données des payloads depuis GitHub ---
def load_payloads_from_github():
    """
    Charge les données des payloads depuis le dépôt GitHub officiel de r-spacex
    """
    url = "https://raw.githubusercontent.com/r-spacex/SpaceX-API/master/docs/payloads/v4/all.json"
    
    try:
        print("🔹 Chargement des données des payloads depuis GitHub...")
        response = requests.get(url, timeout=10)
        
        if response.status_code == 200:
            payloads = response.json()
            # Créer un dictionnaire {id: données_complètes}
            payload_dict = {p['id']: p for p in payloads}
            print(f"✅ {len(payload_dict)} payloads chargés avec succès")
            return payload_dict
        else:
            print(f"❌ Erreur HTTP {response.status_code} - Impossible de charger les payloads")
            return {}
    except Exception as e:
        print(f"❌ Erreur lors du chargement : {e}")
        return {}

# --- 2. Charger les données ---
PAYLOAD_DATA = load_payloads_from_github()

# --- 3. Fonction getPayloadData corrigée ---
def getPayloadData(data):
    """
    Récupère les informations des payloads (masse et orbite) 
    en utilisant les données statiques de GitHub
    """
    global PayloadMass, Orbit
    PayloadMass, Orbit = [], []
    
    if not PAYLOAD_DATA:
        print("⚠️ Aucune donnée de payload disponible. Vérifiez votre connexion.")
        PayloadMass = [None] * len(data)
        Orbit = [None] * len(data)
        return
    
    for load in data['payloads']:
        if load and load in PAYLOAD_DATA:
            payload = PAYLOAD_DATA[load]
            PayloadMass.append(payload.get('mass_kg'))
            Orbit.append(payload.get('orbit'))
        else:
            PayloadMass.append(None)
            Orbit.append(None)
    
    print(f"✅ {len([v for v in PayloadMass if v])} masses de payload récupérées sur {len(PayloadMass)}")
    print(f"✅ {len([v for v in Orbit if v])} orbites récupérées sur {len(Orbit)}")

# --- 4. Appeler la fonction ---
print("\n🔹 Récupération des données des payloads...")
getPayloadData(data)

# --- 5. Afficher les résultats ---
print("\n📋 Aperçu des PayloadMass (10 premiers) :")
print(PayloadMass[:10])
print("\n📋 Aperçu des Orbit (10 premiers) :")
print(Orbit[:10])

# --- 6. Statistiques ---
non_none_mass = [v for v in PayloadMass if v]
non_none_orbit = [v for v in Orbit if v]
print(f"\n📊 Statistiques :")
print(f"  - Total : {len(PayloadMass)}")
print(f"  - Masses récupérées : {len(non_none_mass)}")
print(f"  - Orbites récupérées : {len(non_none_orbit)}")
print(f"  - Orbites uniques : {sorted(set(non_none_orbit))}")

🔹 Chargement des données des payloads depuis GitHub...
❌ Erreur HTTP 404 - Impossible de charger les payloads

🔹 Récupération des données des payloads...
⚠️ Aucune donnée de payload disponible. Vérifiez votre connexion.

📋 Aperçu des PayloadMass (10 premiers) :
[None, None, None, None, None, None, None, None, None, None]

📋 Aperçu des Orbit (10 premiers) :
[None, None, None, None, None, None, None, None, None, None]

📊 Statistiques :
  - Total : 94
  - Masses récupérées : 0
  - Orbites récupérées : 0
  - Orbites uniques : []


In [103]:
# Call getCoreData
# Charger les données des cores depuis GitHub
def load_cores_from_github():
    url = "https://raw.githubusercontent.com/r-spacex/SpaceX-API/master/docs/cores/v4/all.json"
    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            cores = response.json()
            return {c['id']: c for c in cores}
    except:
        pass
    return {}

CORE_DATA = load_cores_from_github()

def getCoreData(data):
    global Block, ReusedCount, Serial, Outcome, Flights, GridFins, Reused, Legs, LandingPad
    Block, ReusedCount, Serial, Outcome, Flights, GridFins, Reused, Legs, LandingPad = [], [], [], [], [], [], [], [], []
    
    for core in data['cores']:
        if core['core'] and core['core'] in CORE_DATA:
            c = CORE_DATA[core['core']]
            Block.append(c.get('block'))
            ReusedCount.append(c.get('reuse_count'))
            Serial.append(c.get('serial'))
        else:
            Block.append(None)
            ReusedCount.append(None)
            Serial.append(None)
        
        Outcome.append(str(core.get('landing_success')) + ' ' + str(core.get('landing_type')))
        Flights.append(core.get('flight'))
        GridFins.append(core.get('gridfins'))
        Reused.append(core.get('reused'))
        Legs.append(core.get('legs'))
        LandingPad.append(core.get('landpad'))
    
    print(f"✅ {len([v for v in Serial if v])} séries de core récupérées")

Finally lets construct our dataset using the data we have obtained. We we combine the columns into a dictionary.


In [104]:
# --- 1. Vérifier que toutes les listes ont la même longueur ---
print("🔍 Vérification des longueurs des listes :")
print(f"  - data (lignes) : {len(data)}")
print(f"  - BoosterVersion : {len(BoosterVersion)}")
print(f"  - LaunchSite : {len(LaunchSite)}")
print(f"  - Longitude : {len(Longitude)}")
print(f"  - Latitude : {len(Latitude)}")
print(f"  - PayloadMass : {len(PayloadMass)}")
print(f"  - Orbit : {len(Orbit)}")
print(f"  - Block : {len(Block)}")
print(f"  - ReusedCount : {len(ReusedCount)}")
print(f"  - Serial : {len(Serial)}")
print(f"  - Outcome : {len(Outcome)}")
print(f"  - Flights : {len(Flights)}")
print(f"  - GridFins : {len(GridFins)}")
print(f"  - Reused : {len(Reused)}")
print(f"  - Legs : {len(Legs)}")
print(f"  - LandingPad : {len(LandingPad)}")

# --- 2. Construire le dictionnaire ---
launch_dict = {
    'FlightNumber': data['flight_number'].values,
    'Date': data['date'].values,
    'BoosterVersion': BoosterVersion,
    'LaunchSite': LaunchSite,
    'Longitude': Longitude,
    'Latitude': Latitude,
    'PayloadMass': PayloadMass,
    'Orbit': Orbit,
    'Block': Block,
    'ReusedCount': ReusedCount,
    'Serial': Serial,
    'Outcome': Outcome,
    'Flights': Flights,
    'GridFins': GridFins,
    'Reused': Reused,
    'Legs': Legs,
    'LandingPad': LandingPad
}

# --- 3. Créer le DataFrame ---
spacex_df = pd.DataFrame(launch_dict)

# --- 4. Afficher les informations ---
print("\n" + "="*60)
print("📊 JEU DE DONNÉES FINAL - SpaceX Launches")
print("="*60)

print(f"\n✅ DataFrame créé avec {len(spacex_df)} lignes et {len(spacex_df.columns)} colonnes.")

print("\n📋 Aperçu des 5 premières lignes :")
print(spacex_df.head())

print("\n📋 Informations sur les colonnes :")
print(spacex_df.info())

print("\n📊 Statistiques descriptives :")
print(spacex_df.describe())

# --- 5. Vérifier les valeurs manquantes ---
print("\n🔍 Valeurs manquantes par colonne :")
print(spacex_df.isnull().sum())

🔍 Vérification des longueurs des listes :
  - data (lignes) : 94
  - BoosterVersion : 94
  - LaunchSite : 94
  - Longitude : 94
  - Latitude : 94
  - PayloadMass : 94
  - Orbit : 94
  - Block : 94
  - ReusedCount : 94
  - Serial : 94
  - Outcome : 94
  - Flights : 94
  - GridFins : 94
  - Reused : 94
  - Legs : 94
  - LandingPad : 94

📊 JEU DE DONNÉES FINAL - SpaceX Launches

✅ DataFrame créé avec 94 lignes et 17 colonnes.

📋 Aperçu des 5 premières lignes :
   FlightNumber        Date BoosterVersion LaunchSite Longitude Latitude  \
0             1  2006-03-24           None       None      None     None   
1             2  2007-03-21           None       None      None     None   
2             4  2008-09-28           None       None      None     None   
3             5  2009-07-13           None       None      None     None   
4             6  2010-06-04           None       None      None     None   

  PayloadMass Orbit Block ReusedCount Serial    Outcome  Flights  GridFins  \
0  

Then, we need to create a Pandas data frame from the dictionary launch_dict.


In [105]:
# Create a data from launch_dict
# Convertir le dictionnaire en DataFrame
spacex_df = pd.DataFrame(launch_dict)

# Afficher les informations
print("✅ DataFrame créé avec succès !")
print(f"📊 {len(spacex_df)} lignes et {len(spacex_df.columns)} colonnes")
print("\n📋 Aperçu des 5 premières lignes :")
print(spacex_df.head())

# Informations sur les colonnes
print("\n📋 Types des colonnes :")
print(spacex_df.dtypes)

# Statistiques descriptives
print("\n📊 Statistiques descriptives :")
print(spacex_df.describe())

# Vérifier les valeurs manquantes
print("\n🔍 Valeurs manquantes par colonne :")
print(spacex_df.isnull().sum())

✅ DataFrame créé avec succès !
📊 94 lignes et 17 colonnes

📋 Aperçu des 5 premières lignes :
   FlightNumber        Date BoosterVersion LaunchSite Longitude Latitude  \
0             1  2006-03-24           None       None      None     None   
1             2  2007-03-21           None       None      None     None   
2             4  2008-09-28           None       None      None     None   
3             5  2009-07-13           None       None      None     None   
4             6  2010-06-04           None       None      None     None   

  PayloadMass Orbit Block ReusedCount Serial    Outcome  Flights  GridFins  \
0        None  None  None        None   None  None None        1     False   
1        None  None  None        None   None  None None        1     False   
2        None  None  None        None   None  None None        1     False   
3        None  None  None        None   None  None None        1     False   
4        None  None  None        None   None  None None     

Show the summary of the dataframe


In [106]:
# Show the head of the dataframe
spacex_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 94 entries, 0 to 93
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   FlightNumber    94 non-null     int64 
 1   Date            94 non-null     object
 2   BoosterVersion  0 non-null      object
 3   LaunchSite      0 non-null      object
 4   Longitude       0 non-null      object
 5   Latitude        0 non-null      object
 6   PayloadMass     0 non-null      object
 7   Orbit           0 non-null      object
 8   Block           0 non-null      object
 9   ReusedCount     0 non-null      object
 10  Serial          0 non-null      object
 11  Outcome         94 non-null     object
 12  Flights         94 non-null     int64 
 13  GridFins        94 non-null     bool  
 14  Reused          94 non-null     bool  
 15  Legs            94 non-null     bool  
 16  LandingPad      64 non-null     object
dtypes: bool(3), int64(2), object(12)
memory usage: 10.7+ KB


### Task 2: Filter the dataframe to only include `Falcon 9` launches


Finally we will remove the Falcon 1 launches keeping only the Falcon 9 launches. Filter the data dataframe using the <code>BoosterVersion</code> column to only keep the Falcon 9 launches. Save the filtered data to a new dataframe called <code>data_falcon9</code>.


In [107]:
# Hint data['BoosterVersion']!='Falcon 1'
data_falcon9 = spacex_df[spacex_df['BoosterVersion'] == 'Falcon 9']

# Afficher les résultats
print("="*60)
print("🚀 LANCEMENTS FALCON 9")
print("="*60)

print(f"\n✅ Nombre de lancements Falcon 9 : {len(data_falcon9)}")
print(f"📊 Nombre de lancements total : {len(spacex_df)}")
print(f"📉 Lancements Falcon 1 supprimés : {len(spacex_df) - len(data_falcon9)}")

print("\n📋 Aperçu des 5 premières lignes :")
print(data_falcon9.head())

print("\n📋 Aperçu des 5 dernières lignes :")
print(data_falcon9.tail())

print("\n🔍 Statistiques des lancements Falcon 9 :")
print(data_falcon9.describe())

print("\n🔍 Distribution des orbites :")
print(data_falcon9['Orbit'].value_counts())

print("\n🔍 Distribution des sites de lancement :")
print(data_falcon9['LaunchSite'].value_counts())


🚀 LANCEMENTS FALCON 9

✅ Nombre de lancements Falcon 9 : 0
📊 Nombre de lancements total : 94
📉 Lancements Falcon 1 supprimés : 94

📋 Aperçu des 5 premières lignes :
Empty DataFrame
Columns: [FlightNumber, Date, BoosterVersion, LaunchSite, Longitude, Latitude, PayloadMass, Orbit, Block, ReusedCount, Serial, Outcome, Flights, GridFins, Reused, Legs, LandingPad]
Index: []

📋 Aperçu des 5 dernières lignes :
Empty DataFrame
Columns: [FlightNumber, Date, BoosterVersion, LaunchSite, Longitude, Latitude, PayloadMass, Orbit, Block, ReusedCount, Serial, Outcome, Flights, GridFins, Reused, Legs, LandingPad]
Index: []

🔍 Statistiques des lancements Falcon 9 :
       FlightNumber  Flights
count           0.0      0.0
mean            NaN      NaN
std             NaN      NaN
min             NaN      NaN
25%             NaN      NaN
50%             NaN      NaN
75%             NaN      NaN
max             NaN      NaN

🔍 Distribution des orbites :
Series([], Name: Orbit, dtype: int64)

🔍 Distribution

Now that we have removed some values we should reset the FlgihtNumber column


In [109]:
data_falcon9.loc[:,'FlightNumber'] = list(range(1, data_falcon9.shape[0]+1))
data_falcon9

,FlightNumber,Date,BoosterVersion,LaunchSite,Longitude,Latitude,PayloadMass,Orbit,Block,ReusedCount,Serial,Outcome,Flights,GridFins,Reused,Legs,LandingPad


## Data Wrangling


We can see below that some of the rows are missing values in our dataset.


In [110]:
data_falcon9.isnull().sum()

FlightNumber      0.0
Date              0.0
BoosterVersion    0.0
LaunchSite        0.0
Longitude         0.0
Latitude          0.0
PayloadMass       0.0
Orbit             0.0
Block             0.0
ReusedCount       0.0
Serial            0.0
Outcome           0.0
Flights           0.0
GridFins          0.0
Reused            0.0
Legs              0.0
LandingPad        0.0
dtype: float64

Before we can continue we must deal with these missing values. The <code>LandingPad</code> column will retain None values to represent when landing pads were not used.


### Task 3: Dealing with Missing Values


Calculate below the mean for the <code>PayloadMass</code> using the <code>.mean()</code>. Then use the mean and the <code>.replace()</code> function to replace `np.nan` values in the data with the mean you calculated.


In [114]:
# --- 1. Convertir la colonne en type numérique ---
# Convertir les None en NaN et forcer le type en float
data_falcon9['PayloadMass'] = pd.to_numeric(data_falcon9['PayloadMass'], errors='coerce')

# Vérifier le type après conversion
print(f"Type après conversion : {data_falcon9['PayloadMass'].dtype}")

# --- 2. Calculer la moyenne (en ignorant les NaN) ---
payload_mean = data_falcon9['PayloadMass'].mean()
print(f"📊 Moyenne de PayloadMass : {payload_mean:.2f} kg")

# --- 3. Remplacer les NaN par la moyenne ---
data_falcon9['PayloadMass'] = data_falcon9['PayloadMass'].fillna(payload_mean)

# --- 4. Vérifier ---
missing_after = data_falcon9['PayloadMass'].isnull().sum()
print(f"\n🔍 Valeurs manquantes après remplacement : {missing_after}")

# --- 5. Afficher les statistiques ---
print("\n📊 Statistiques de PayloadMass après remplacement :")
print(data_falcon9['PayloadMass'].describe())

Type après conversion : int64
📊 Moyenne de PayloadMass : nan kg

🔍 Valeurs manquantes après remplacement : 0

📊 Statistiques de PayloadMass après remplacement :
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: PayloadMass, dtype: float64


You should see the number of missing values of the <code>PayLoadMass</code> change to zero.


Now we should have no missing values in our dataset except for in <code>LandingPad</code>.


We can now export it to a <b>CSV</b> for the next section,but to make the answers consistent, in the next lab we will provide data in a pre-selected date range. 


<code>data_falcon9.to_csv('dataset_part_1.csv', index=False)</code>


In [115]:
# --- Diagnostic complet ---
print("="*60)
print("🔍 DIAGNOSTIC DE data_falcon9")
print("="*60)

print(f"\n1. Nombre de lignes dans data_falcon9 : {len(data_falcon9)}")
print(f"2. Colonnes disponibles : {data_falcon9.columns.tolist()}")

if len(data_falcon9) == 0:
    print("\n⚠️ data_falcon9 est vide !")
    print("   Cela explique pourquoi la moyenne est NaN.")
    print("   Vérifiez que le filtrage Falcon 9 a fonctionné.")
else:
    # Afficher les 5 premières valeurs de PayloadMass
    print(f"\n3. Aperçu de PayloadMass (10 premières) :")
    print(data_falcon9['PayloadMass'].head(10))
    
    # Vérifier les valeurs uniques
    print(f"\n4. Valeurs uniques dans PayloadMass :")
    print(data_falcon9['PayloadMass'].unique())
    
    # Vérifier les types
    print(f"\n5. Type de PayloadMass : {data_falcon9['PayloadMass'].dtype}")
    
    # Vérifier les valeurs manquantes
    print(f"\n6. Nombre de NaN : {data_falcon9['PayloadMass'].isna().sum()}")
    
    # Vérifier si toutes les valeurs sont None/NaN
    non_null = data_falcon9['PayloadMass'].notna().sum()
    print(f"7. Valeurs non-NaN : {non_null} sur {len(data_falcon9)}")
    
    if non_null == 0:
        print("\n⚠️ Toutes les valeurs sont manquantes !")
        print("   Il faut charger les données de PayloadMass depuis GitHub.")

🔍 DIAGNOSTIC DE data_falcon9

1. Nombre de lignes dans data_falcon9 : 0
2. Colonnes disponibles : ['FlightNumber', 'Date', 'BoosterVersion', 'LaunchSite', 'Longitude', 'Latitude', 'PayloadMass', 'Orbit', 'Block', 'ReusedCount', 'Serial', 'Outcome', 'Flights', 'GridFins', 'Reused', 'Legs', 'LandingPad']

⚠️ data_falcon9 est vide !
   Cela explique pourquoi la moyenne est NaN.
   Vérifiez que le filtrage Falcon 9 a fonctionné.


## Authors


<a href="https://www.linkedin.com/in/joseph-s-50398b136/">Joseph Santarcangelo</a> has a PhD in Electrical Engineering, his research focused on using machine learning, signal processing, and computer vision to determine how videos impact human cognition. Joseph has been working for IBM since he completed his PhD. 


<!--## Change Log
-->


<!--

|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2020-09-20|1.1|Joseph|get result each time you run|
|2020-09-20|1.1|Azim |Created Part 1 Lab using SpaceX API|
|2020-09-20|1.0|Joseph |Modified Multiple Areas|
-->


Copyright ©IBM Corporation. All rights reserved.
